In [10]:
import cobra

import pandas as pd

from Bio.Seq import Seq
from Bio import SeqIO
from Bio.Alphabet import generic_dna

import multiprocessing
from multiprocessing import Pool
# from tqdm import tqdm

import scipy.stats as st
from bs4 import BeautifulSoup
import urllib
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings

import requests, sys, json, re
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *


In [11]:
# MANE SELECTED transcripts and protein sequences
ids = pd.read_csv(local_data_path + 'raw/MANE.GRCh38.v0.9.summary.txt', sep = '\t')

psim_me = ids.loc[:, ['Ensembl_Gene', 'HGNC_ID', '#NCBI_GeneID', 'Ensembl_nuc', 'Ensembl_prot', 'symbol', 'name', 'chr_strand']]
psim_me.columns = ['ENSG_ID', 'HGNC_ID', 'NCBI_ID', 'ENST_ID', 'ENSP_ID', 'GENE_SYMBOL', 'GENE_NAME', 'CHR_STRAND']

polyA = pd.read_csv(local_data_path + 'processed/polyA_length.csv', index_col = 0)
psim_me['POLYA_LENGTH'] = psim_me.GENE_SYMBOL.map(dict(zip(polyA.index.tolist(), polyA.MEAN.tolist())))

sequence mapping

In [ ]:
# add protein sequences
protein = list(SeqIO.parse(local_data_path + 'raw/MANE.GRCh38.v0.9.select_ensembl_protein.faa', "fasta"))
p_map = dict()
for p in protein:
    p_map[p.id] = str(p.seq)
psim_me['PROTEIN_SEQ'] = psim_me.ENSP_ID.map(p_map)

# add mrna sequence
mrna = list(SeqIO.parse(local_data_path + 'raw/MANE.GRCh38.v0.9.select_ensembl_rna.fna', "fasta"))
m_map = dict()
for m in mrna:
    m_map[m.id] = str(m.seq)
psim_me['MRNA_SEQ'] = psim_me.ENST_ID.map(m_map)

premrna more complicated because FTP doesn't have ENSG to full gene sequence

In [ ]:
# # # add premrna sequence - no FTP file for this, parallelize REST API instead
# def get_premrna_seq(ensg_id, counter):
#     print(counter)
#     try:
#         hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
#         return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text # introns and UTRs
#     except:
#         return float('nan')

# # pool = Pool(processes = multiprocessing.cpu_count())
# # premrna = pool.starmap(get_premrna_seq, zip(psim_me['ENSG_ID'].apply(lambda x: x.split('.')[0]).tolist(), list(range(psim_me.shape[0]))))
# # pool.close()

# # with open(local_data_path + 'interim/premrna_sequences.txt', 'w') as f:
# #     for seq in premrna:
# #         if type(seq) != str:
# #             seq = str(seq)
# #         f.write(seq + '\n')

# premrna = open(local_data_path + 'interim/premrna_sequences.txt', 'r').read().splitlines()
###NOT ALL DOWNLOADED, SO RERUNNING ON THOSE THAT DIDNT DOWNLOAD
# fail = 'You have exceeded the limit of 15 requests per second; please reduce your concurrent connections'
# fail_index = [i for i in range(len(premrna)) if premrna[i] == fail]
# pool = Pool(processes = 4)
# corrected = pool.starmap(get_premrna_seq, zip(psim_me.loc[fail_index, 'ENSG_ID'].apply(lambda x: x.split('.')[0]).tolist(), list(range(psim_me.shape[0]))))
# pool.close()
# for idx, seq in dict(zip(fail_index, corrected)).items():
#     premrna[idx] = seq

# with open(local_data_path + 'interim/premrna_sequences_v2.txt', 'w') as f:
#     for seq in premrna:
#         if type(seq) != str:
#             seq = str(seq)
#         f.write(seq + '\n')
premrna2 = open(local_data_path + 'interim/premrna_sequences_v2.txt', 'r').read().splitlines()
psim_me['PREMRNA_SEQ'] = premrna2

psim_me.loc[psim_me[psim_me.PREMRNA_SEQ == 'nan'].index, 'PREMRNA_SEQ'] = float('nan')

for i in psim_me.index:
    if type(psim_me.loc[i, 'PREMRNA_SEQ']) == str:
        if len(psim_me.loc[i,'MRNA_SEQ']) > len(psim_me.loc[i,'PREMRNA_SEQ']):
            psim_me.loc[i,'PREMRNA_SEQ'] = float('nan')
            

In [ ]:
# formatting
def transcribe(x):
    try: 
        return str(Seq(x).transcribe())
    except:
        return float('nan')

psim_me['MRNA_SEQ'] = psim_me['MRNA_SEQ'].apply(lambda x: transcribe(x))
psim_me['PREMRNA_SEQ'] = psim_me['PREMRNA_SEQ'].apply(lambda x: transcribe(x))

uniprot_map = pd.read_csv(local_data_path + 'raw/gencode.v34.metadata.SwissProt', sep = '\t', header = None)
psim_me['UNIPROT_ID'] = psim_me['ENST_ID'].map(dict(zip(uniprot_map[0].tolist(), uniprot_map[1].tolist())))

psim_human = pd.read_csv(root_path + 'MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/PSIM_HUMAN.tab', 
                         sep = '\t')

cols = ['SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'Location']
for col in cols:
    psim_me[col] = psim_me['UNIPROT_ID'].map(dict(zip(psim_human.Entry.tolist(), psim_human[col].tolist())))

for i in psim_me.Location.dropna().index:
    a = psim_me.loc[i, 'Location']
    psim_me.loc[i, 'Location'] = a.split('[')[1].split(']')[0]

# To Do

For machinery that don't have a MANE transcript, develop an alternative method to get the PSIM information. For now, I am taking the first result from the ENSEMBL rest api to build some of the necessary reactions but later will change this to the general method. 

In [39]:
def get_premrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=genomic' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

def get_mrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=cdna;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

def get_protein_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=protein;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')
    
def get_all_seq(ensg_id):
    return get_premrna_seq(ensg_id), get_mrna_seq(ensg_id).splitlines()[0], get_protein_seq(ensg_id).splitlines()[0]

In [85]:
# psim_me = pd.read_csv(local_data_path + 'processed/psim_me.csv', index_col = 0)


In [129]:
# RPS27a
cols = ['ENSG_ID', 'HGNC_ID', 'GENE_SYMBOL', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'POLYA_LENGTH']
ensg_id, hgnc_id, gene_name = 'ENSG00000143947', 'HGNC:10417', 'RPS27A'
premrna, mrna, protein = get_all_seq(ensg_id)
polyA_L = polyA.loc[gene_name,'MEAN']
psim_me.loc[psim_me.shape[0],:] = [float('nan')]*psim_me.shape[1]
psim_me.loc[psim_me.shape[0]-1,cols] = [ensg_id, hgnc_id, gene_name, protein, transcribe(mrna), 
                                        transcribe(premrna),polyA_L]

# RPLs
cols = ['ENSG_ID', 'HGNC_ID', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'POLYA_LENGTH']
rl_missing_hgnc = ['HGNC:10307', 'HGNC:10313', 'HGNC:10340', 'HGNC:10362', 'HGNC:10368']
rl_missing_ensg = ['ENSG00000265681', 'ENSG00000122026', 'ENSG00000109475', 'ENSG00000089009', 'ENST00000262584']
for i in range(len(rl_missing_hgnc)):
    ensg_id, hgnc_id = rl_missing_ensg[i], rl_missing_hgnc[i]
    premrna, mrna, protein = get_all_seq(ensg_id)
    try:
        polyA_L = polyA.loc[gene_name,'MEAN']
    except:
        polyA_l = float('nan')
    
    psim_me.loc[psim_me.shape[0],:] = [float('nan')]*psim_me.shape[1]
    psim_me.loc[psim_me.shape[0]-1,cols] = [ensg_id, hgnc_id, protein, transcribe(mrna), 
                                            transcribe(premrna),polyA_L]

In [136]:
psim_me.to_csv(local_data_path + 'processed/psim_me.csv')

# You are here
must fill out missing machinery sequences

In [ ]:
metabolic_machinery = sorted(set([gene.id for gene in human_model.genes]))

# map sec machinery rxn GPRs from entrez to HGNC IDs
rxnGPR = open(root_path + "MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/rxnGPRs_HUMAN.txt").read().splitlines()
line1 = rxnGPR[0]
rxnGPR = rxnGPR[1:]
secretory_machinery = []
for i in rxnGPR:
    secretory_machinery += re.findall(r'\d+', i)
secretory_machinery = sorted(set(secretory_machinery))
entrez_hgnc_map = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
entrez_hgnc_map = entrez_hgnc_map.loc[entrez_hgnc_map['NCBI gene ID'].dropna().index,:]
entrez_hgnc_map['NCBI gene ID'] = entrez_hgnc_map['NCBI gene ID'].astype('int64').astype(str)
entrez_hgnc_map = entrez_hgnc_map[entrez_hgnc_map['NCBI gene ID'].isin(secretory_machinery)]
secretory_machinery = entrez_hgnc_map['HGNC ID'].tolist()

entrez_hgnc_map = dict(zip(entrez_hgnc_map['NCBI gene ID'].tolist(),entrez_hgnc_map['HGNC ID'].tolist()))

for i in range(len(rxnGPR)):
    mach = re.findall(r'\d+', rxnGPR[i])
    for m in mach:
        rxnGPR[i] = rxnGPR[i].replace(m, entrez_hgnc_map[m])

rxnGPR = [line1] + rxnGPR

with open(local_data_path + 'processed/rxnGPRs_HUMAN_HGNCID.txt', 'w') as f:
    for i in rxnGPR:
        f.write(i + '\n')

a = len(set(metabolic_machinery))
b = len(set(metabolic_machinery).difference(psim_me.loc[:, ['PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'HGNC_ID']].dropna().HGNC_ID.tolist()))

print('{} of {} RECON2.2 machinery mapped'.format(a-b, a))




a = len(set(secretory_machinery))
b = len(set(secretory_machinery).difference(psim_me.loc[:,['PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'HGNC_ID']].dropna().NCBI_ID.apply(lambda x: x.split('GeneID:')[1]).tolist()))

print('{} of {} secretory machinery mapped'.format(a-b, a))

In [ ]:
psim_me.to_csv(local_data_path + 'processed/psim_me.csv')